Business question #1: Financial Health & True Profitability
Does discounting actually pay for itself?

In [20]:
With discount_buckets as 
(
    SELECT CASE WHEN discount_pct = 0 THEN '0% - No Discount'
            WHEN discount_pct <= 0.10 THEN '1-10%'
            WHEN discount_pct <= 0.20 THEN '11-20%'
            WHEN discount_pct <= 0.30 THEN '21-30%'
            WHEN discount_pct <= 0.40 THEN '31-40%'
            WHEN discount_pct <= 0.50 THEN '41-50%'
            WHEN discount_pct <= 0.60 THEN '51-60%'
            WHEN discount_pct <= 0.70 THEN '61-70%'
            WHEN discount_pct <= 0.80 THEN '71-80%'
            ELSE '80%+'
            END AS discount_tiers,
            quantity,
            net_revenue,
            gross_profit,
            gross_margin_pct,
            product_id
    from FactSales
            
)

SELECT  count(DISTINCT product_id) as product_count,
    discount_tiers,

    sum(quantity) AS Orders,
    sum(net_revenue) as Revenue,
    sum(gross_profit) as Profit,
    AVG(gross_margin_pct) as gross_margin_pct
FROM discount_buckets
group by discount_tiers

(10 rows affected)

product_count | discount_tiers   | Orders | Revenue            | Profit             | gross_margin_pct
--------------+------------------+--------+--------------------+--------------------+-----------------
83            | 0% - No Discount | 842321 | 126255313.73001935 | 72879042.84000653  | 0.585274        
83            | 1-10%            | 268063 | 36655489.880001515 | 19401616.6499986   | 0.539042        
83            | 11-20%           | 64021  | 7637142.120000074  | 3560602.649999962  | 0.477646        
83            | 21-30%           | 66051  | 6912028.690000074  | 2749110.830000003  | 0.408184        
83            | 31-40%           | 69210  | 6342430.900000102  | 2008131.5800000161 | 0.307708        
82            | 41-50%           | 5180   | 751139.2300000023  | 421402.35000000155 | 0.552260        
83            | 51-60%           | 5384   | 682920.7699999999  | 387408.88000000076 | 0.561452        
82            | 61-70%           | 4860   | 632266.41

Key Findings
An analysis of discount percentage against volume and gross margin reveals three critical insights regarding our Q2 pricing strategy. First, discounts are failing to drive incremental demand. Contrary to standard economic expectations, order volume drops by 68% the moment a discount is introduced (from 842k at 0% to 268k at 1-10%), and remains stagnant around 64k-69k orders through the 11-40% tiers. This indicates we are applying discounts to slow-moving inventory rather than stimulating new purchases. 

Second, because volume does not increase to offset the price reduction, we are unequivocally giving money away. Gross margin steadily degrades from 58.5% (at 0%) to a trough of 30.7% (at 31-40%), causing absolute profit to plummet from $72.8M down to just $2.0M. 





Executive Summary
Our current discounting strategy is not paying for itself; instead, it is actively eroding profitability without generating incremental volume. The 0% discount tier is the overwhelming engine of the business, responsible for 842k orders and $72.8M in profit. Conversely, mid-tier discounts (11-30%) suffer a "double whammy" of falling demand and collapsing margins, proving that the extra quantity does not offset the margin hit. Furthermore, a suspicious margin spike in the 41%+ discount tiers reveals that we are not comparing like-for-like products, rendering standard pricing rules ineffective for that segment. We recommend an immediate reduction in broad promotional discounts and a strategic audit of the 41%+ product mix.

business question #2: Product Performance & Trends
Which products are driving volume vs. driving profit?

In [32]:
With cat as 
(
    SELECT  product_id,
        CASE 
        WHEN SUM(quantity) >= PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY sum(quantity)) over() and 
            AVG(gross_margin_pct) >= PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY avg(gross_margin_pct)) over() THEN 'High Volume/High Margin (Heroes)'
        WHEN SUM(quantity) >= PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY sum(quantity)) over() and 
            AVG(gross_margin_pct) <= PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY avg(gross_margin_pct)) over() THEN 'High Volume/Low Margin (Trafic Drivers)'
        WHEN SUM(quantity) <= PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY sum(quantity)) over() and 
            AVG(gross_margin_pct) >= PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY avg(gross_margin_pct)) over() THEN 'Low Volume/High Margin (Hidden Gems)'
        WHEN SUM(quantity) <= PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY sum(quantity)) over() and 
            AVG(gross_margin_pct) <= PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY avg(gross_margin_pct)) over() THEN 'Low Volume/Low Margin (Dead Weight).'
        
        else null 
        END as product_cat 
    from FactSales
    GROUP BY product_id
)
select count (product_id) as Products, product_cat
from cat
GROUP BY product_cat

(4 rows affected)

Products | product_cat                            
---------+----------------------------------------
24       | High Volume/High Margin (Heroes)       
18       | High Volume/Low Margin (Trafic Drivers)
18       | Low Volume/High Margin (Hidden Gems)   
23       | Low Volume/Low Margin (Dead Weight).   
(4 rows)

Total execution time: 00:00:04.880

Key Findings
A quadrant analysis of our 83 products reveals a highly actionable distribution across the four performance tiers, but it is the interplay between these categories that presents the real strategic opportunity.

The most urgent finding is that "Dead Weight" (Low Volume/Low Margin) is our single largest category at 23 products. This represents wasted supply chain effort, warehousing costs, and marketing spend; these items should be flagged for immediate discontinuation or aggressive liquidation. Conversely, our 18 "Hidden Gems" (Low Volume/High Margin) represent our highest-leverage assets. Because these products already prove customers will pay a premium for them, their low volume is likely a visibility or awareness issue. The merchandising team should immediately bundle or cross-sell these Hidden Gems alongside our 24 "Heroes" to drive volume without sacrificing margin.

Finally, the 18 "Traffic Drivers" (High Volume/Low Margin) require an immediate pricing audit. Given our earlier finding that discounting destroys profit without actually driving incremental volume, it is highly probable that these 18 products are the ones stuck in the 11-30% discount tiers. By tightening the promotional spend on these Traffic Drivers and reining in their discounts, we have a clear pathway to migrate them directly into the "Hero" quadrant, dramatically increasing overall portfolio profitability.

Our 83-product portfolio is severely bloated, with over a quarter of our catalog classified as "Dead Weight" that generates neither volume nor margin. While our 24 "Hero" products form a healthy, profitable core, we are leaving significant money on the table. We are failing to convert our 18 "Hidden Gems" into high-volume sellers through proper marketing, and we are allowing 18 "Traffic Drivers" to cannibalize shelf space with sub-optimal margins. Immediate action is required to rationalize the bottom 23 products and strategically reallocate merchandising focus to unlock the high-margin "Hidden Gems."



Customer Behavior & Value 
What is our customer repeat purchase rate?

In [47]:
WITH count as 
(
    SELECT customer_id,count(distinct order_id) as orders,YEAR(order_date) as Year
    from FactSales
    GROUP BY YEAR(order_date), customer_id
)

SELECT 100.0 * count(case when orders > 1 then 1 end) / 
    count(*) as repeat_purchase_rate,Year
from count 
where year !=  2025
GROUP BY year




(3 rows affected)

repeat_purchase_rate | Year
---------------------+-----
26.444294192772      | 2022
25.907280787437      | 2023
26.914083125895      | 2024
(3 rows)

Total execution time: 00:00:05.091

In [48]:
WITH count as 
(
    SELECT customer_id,count(distinct order_id) as orders
    from FactSales
    GROUP BY customer_id
)

SELECT 100.0 * count(case when orders > 1 then 1 end) / 
    count(*) as repeat_purchase_rate
from count 





(1 row affected)

repeat_purchase_rate
--------------------
63.602394632166     
(1 row)

Total execution time: 00:00:04.459

Key Findings
The most critical insight in this data is not the numbers themselves, but the stark divergence between the overall repeat rate (63.6%) and the yearly repeat rate (~26%). The 63.6% figure represents lifetime retention customers buying in 2022 and returning in 2024 proving the underlying product has strong staying power. However, the yearly view reveals that same-year repurchase rates are locked in a tight, stagnant band between 25.9% and 26.9%.

This flatline tells us two things. First, our post-purchase nurture campaigns (like automated email flows or 30-day retargeting) have hit a hard ceiling; whatever they are doing is yielding the exact same result year over year. Second, this behavior strongly implies our products have a longer replacement or consumption cycle. Customers simply do not need to buy our core items again within 12 months. To improve same-year frequency, the marketing team must stop using generic "buy again" prompts and instead pivot to lifecycle triggers such as cross-selling complementary items from the "Hidden Gems" quadrant, seasonal refreshes, or tiered loyalty programs—rather than fighting against the natural usage cycle of the product.

Executive summary: 

Our customer base exhibits strong long-term loyalty with a lifetime repeat purchase rate of 63.6% but struggles with short-term, same-year frequency. Only about 26% of customers make more than one purchase within a single calendar year, a metric that has remained completely stagnant from 2022 through 2024. This indicates that while our brand retains customers over a multi-year horizon, our current retention marketing is failing to accelerate purchases within a 12-month window, likely pointing to a longer natural purchase cycle for our products.



business question #4 Operations & Fulfillment
Where's fulfillment breaking down? (merge their Q6 + Q7) — Combine order-to-ship-to-delivery time with the product/region breakdown into one question instead of two. Genuinely new territory — neither existing project touches operations. Fix the ship_date duplication above before building this one.


In [ ]:
SELECT fs.order_id, fs.product_id,country_normalized,
    DATEDIFF(day,order_date, ship_date) as processing_days,
    DATEDIFF(day,ship_date, delivery_date) as transit_days,
    DATEDIFF(day,order_date, delivery_date) as total_fulfillment_days

from FactSales fs
LEFT JOIN FactReturns fr on fs.geography_key = fr.geography_key
LEFT JOIN DimGeography dg on fs.geography_key = dg.geography_key
where    DATEDIFF(day,order_date, ship_date)  not between 0 and 30

Msg 209, Level 16, State 1, Line 1
Ambiguous column name 'order_id'.
Msg 209, Level 16, State 1, Line 1
Ambiguous column name 'product_id'.

Total execution time: 00:00:00.177

In [49]:
SELECT top 5 *
FROM DimGeography

(5 rows affected)

geography_key | region | country_normalized
--------------+--------+-------------------
1             | BC     | Canada            
2             | ON     | Canada            
3             | QC     | Canada            
4             | ENG    | United Kingdom    
5             | SCT    | United Kingdom    
(5 rows)

Total execution time: 00:00:01.712